In [8]:
import torch
import math

In [13]:
class ScratchEncoder:
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        combined_size = input_size + hidden_size
        
        # Helper function for weight initialization
        def init_weight(rows, cols):
            stdv = 1.0 / math.sqrt(hidden_size)
            tensor = torch.empty(rows, cols).uniform_(-stdv, stdv)
            tensor.requires_grad = True
            return tensor
            
        def init_bias(size):
            tensor = torch.zeros(size)
            tensor.requires_grad = True
            return tensor

        # EXACT WHITEBOARD VARIABLES
        self.W_f = init_weight(hidden_size, combined_size)
        self.b_f = init_bias(hidden_size)
        
        self.W_i = init_weight(hidden_size, combined_size)
        self.b_i = init_bias(hidden_size)
        
        self.W_k = init_weight(hidden_size, combined_size) # 'k' from your board
        self.b_k = init_bias(hidden_size)
        
        self.W_o = init_weight(hidden_size, combined_size)
        self.b_o = init_bias(hidden_size)
        
        # We will cache states here later for the manual BPTT!
        self.cache = []

    def forward(self, X):
            # X shape: (batch_size, seq_len, input_size)
            batch_size, seq_len, _ = X.shape
            
            h_t = torch.zeros(batch_size, self.hidden_size)
            c_t = torch.zeros(batch_size, self.hidden_size)
            
            for t in range(seq_len):
                x_t = X[:, t, :]
                
                # Concatenate h and x (your hx_t)
                hx_t = torch.cat((h_t, x_t), dim=1)
                
                # The Whiteboard Math
                f = torch.sigmoid(hx_t @ self.W_f.T + self.b_f)
                i = torch.sigmoid(hx_t @ self.W_i.T + self.b_i)
                k = torch.tanh(hx_t @ self.W_k.T + self.b_k)
                o = torch.sigmoid(hx_t @ self.W_o.T + self.b_o)
                
                c_t = (f * c_t) + (i * k)
                h_t = torch.tanh(c_t) * o
                
            # ---> MAKE SURE THIS LINE IS HERE! <---
            # The Encoder ONLY returns the final context vectors
            return h_t, c_t

In [14]:
class ScratchDecoder:
    def __init__(self, input_size, hidden_size, vocab_size):
        self.hidden_size = hidden_size
        combined_size = input_size + hidden_size
        
        def init_weight(rows, cols):
            stdv = 1.0 / math.sqrt(hidden_size)
            tensor = torch.empty(rows, cols).uniform_(-stdv, stdv)
            tensor.requires_grad = True
            return tensor
            
        def init_bias(size):
            tensor = torch.zeros(size)
            tensor.requires_grad = True
            return tensor

        # DECODER'S INDEPENDENT WEIGHTS
        self.W_f = init_weight(hidden_size, combined_size)
        self.b_f = init_bias(hidden_size)
        
        self.W_i = init_weight(hidden_size, combined_size)
        self.b_i = init_bias(hidden_size)
        
        self.W_k = init_weight(hidden_size, combined_size)
        self.b_k = init_bias(hidden_size)
        
        self.W_o = init_weight(hidden_size, combined_size)
        self.b_o = init_bias(hidden_size)
        
        # FC Layer: y^ = FC(h') from your board
        self.W_fc = init_weight(vocab_size, hidden_size)
        self.b_fc = init_bias(vocab_size)
        
        self.cache = []

    def forward(self, y_t, h_prev, c_prev):
        # y_t is the CURRENT target word embedding. Shape: (batch_size, input_size)
        
        # Concatenate previous h and current y (your hy_t)
        hy_t = torch.cat((h_prev, y_t), dim=1)
        
        # The Whiteboard Math
        f = torch.sigmoid(hy_t @ self.W_f.T + self.b_f)
        i = torch.sigmoid(hy_t @ self.W_i.T + self.b_i)
        k = torch.tanh(hy_t @ self.W_k.T + self.b_k)
        o = torch.sigmoid(hy_t @ self.W_o.T + self.b_o)
        
        c_next = (f * c_prev) + (i * k)
        h_next = torch.tanh(c_next) * o
        
        # y^ = FC(h')
        y_hat = h_next @ self.W_fc.T + self.b_fc
        
        return y_hat, h_next, c_next

In [15]:
class ScratchSeq2Seq:
    def __init__(self, encoder, decoder):
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, X, Y_embeddings):
        # X shape: (batch_size, source_seq_len, input_size)
        # Y_embeddings shape: (batch_size, target_seq_len, input_size)
        
        batch_size, target_seq_len, _ = Y_embeddings.shape
        vocab_size = self.decoder.W_fc.shape[0]
        
        # Tensor to store all our \hat{y} predictions
        predictions = torch.zeros(batch_size, target_seq_len, vocab_size)
        
        # 1. ENCODER PASS: Compress the entire source sequence
        context_h, context_c = self.encoder.forward(X)
        
        # 2. THE BRIDGE: Initialize the Decoder with the Encoder's Context Vector
        decoder_h = context_h
        decoder_c = context_c
        
        # 3. DECODER PASS: Unroll the target sequence
        for t in range(target_seq_len):
            # In a real training loop, we use Teacher Forcing here
            y_t = Y_embeddings[:, t, :] 
            
            # Get the prediction for this step
            y_hat, decoder_h, decoder_c = self.decoder.forward(y_t, decoder_h, decoder_c)
            
            # Save the prediction
            predictions[:, t, :] = y_hat
            
        return predictions

In [16]:
# 1. Define our experimental dimensions
batch_size = 4          # Processing 4 sentences at once
source_seq_len = 7      # e.g., A 7-word English sentence
target_seq_len = 9      # e.g., A 9-word French translation
embed_dim = 16
# The size of our word vectors
hidden_size = 32        # The size of our LSTM memory (C and h)
vocab_size = 5000       # How many words our Decoder can choose from

encoder = ScratchEncoder(input_size=embed_dim, hidden_size=hidden_size)
decoder = ScratchDecoder(input_size=embed_dim, hidden_size=hidden_size, vocab_size=vocab_size)
seq2seq = ScratchSeq2Seq(encoder, decoder)

# 3. Generate dummy data (Normally, this comes from an Embedding layer)
# X represents the English sentence (7 words long)
X_dummy = torch.randn(batch_size, source_seq_len, embed_dim)

# Y represents the French words we feed the Decoder during Teacher Forcing (9 words long)
Y_dummy = torch.randn(batch_size, target_seq_len, embed_dim)

# 4. Run the Forward Pass!
print("\n--- Running the Forward Pass ---")
predictions = seq2seq.forward(X_dummy, Y_dummy)

# 5. Verify the architecture's output
print(f"Input X shape:           {X_dummy.shape}   -> (batch, source_len, embed_dim)")
print(f"Target Y shape:          {Y_dummy.shape}   -> (batch, target_len, embed_dim)")
print(f"Final Predictions shape: {predictions.shape} -> (batch, target_len, vocab_size)")

--- Initializing the Seq2Seq Model ---

--- Running the Forward Pass ---
Input X shape:           torch.Size([4, 7, 16])   -> (batch, source_len, embed_dim)
Target Y shape:          torch.Size([4, 9, 16])   -> (batch, target_len, embed_dim)
Final Predictions shape: torch.Size([4, 9, 5000]) -> (batch, target_len, vocab_size)
